In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pymargins import Margins, SurveyDesign

DATA = "../demos/data"
pop = pd.read_csv(f"{DATA}/apipop.csv")
strat = pd.read_csv(f"{DATA}/apistrat.csv")
clus = pd.read_csv(f"{DATA}/apiclus1.csv")

print(f"population: {len(pop):>5} schools")
print(f"stratified: {len(strat):>5} schools (strata = school type)")
print(f"cluster:    {len(clus):>5} schools in "
      f"{clus['dnum'].nunique()} districts")

population:  6194 schools
stratified:   200 schools (strata = school type)
cluster:      183 schools in 15 districts


In [2]:
TRUTH_MEAN = pop["api00"].mean()
truth_fit = smf.ols("api00 ~ meals + ell", data=pop).fit()
TRUTH_AME = truth_fit.params["meals"]

print(f"population mean api00:      {TRUTH_MEAN:7.2f}")
print(f"population slope on meals:  {TRUTH_AME:7.3f}")

population mean api00:       664.71
population slope on meals:   -2.963


In [3]:
mix = pd.DataFrame({
    "population %": 100 * pop["stype"].value_counts(normalize=True),
    "sample %": 100 * strat["stype"].value_counts(normalize=True),
    "weight pw": strat.groupby("stype")["pw"].first(),
}).round(1)
print(mix)

       population %  sample %  weight pw
stype                                   
E              71.4      50.0       44.2
H              12.2      25.0       15.1
M              16.4      25.0       20.4


In [4]:
unweighted = strat["api00"].mean()
weighted = np.average(strat["api00"], weights=strat["pw"])

print(f"unweighted sample mean: {unweighted:7.2f}")
print(f"weighted sample mean:   {weighted:7.2f}")
print(f"population truth:        {TRUTH_MEAN:7.2f}")

unweighted sample mean:  652.82
weighted sample mean:    662.29
population truth:         664.71


In [5]:
fit_w = smf.glm(
    "api00 ~ meals + ell",
    data=strat,
    freq_weights=strat["pw"].values,
).fit()

m_weighted = Margins(fit_w, at="overall", weights=strat["pw"].values)
print(f"weighted population-mean prediction: "
      f"{float(m_weighted.predict().estimate):.2f}")

weighted population-mean prediction: 662.29


In [6]:
# SurveyDesign wants integer-coded strata
stratum_codes = strat["stype"].astype("category").cat.codes.values

design = SurveyDesign(
    weights=strat["pw"].values,
    strata=stratum_codes,
    fpc=strat["fpc"].values,
)

m_survey = Margins(
    fit_w, survey_design=design, weights=strat["pw"].values, at="overall"
)
print(m_survey.dydx("meals").summary())

             Margins Result (delta, level=0.95)            
       estimate  std err         z  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
meals   -3.1106   0.2758  -11.2800  0.000  -3.6511, -2.5701

n = 200
κ: 0.000
Delta-vs-sim disagreement: 0.958%


In [7]:
fit_c = smf.glm(
    "api00 ~ meals + ell",
    data=clus,
    freq_weights=clus["pw"].values,
).fit()

m_naive = Margins(fit_c, at="overall", weights=clus["pw"].values)

design_c = SurveyDesign(
    weights=clus["pw"].values,
    psu=clus["dnum"].values,     # primary sampling unit = school district
    fpc=clus["fpc"].values,
)
m_cluster = Margins(
    fit_c, survey_design=design_c, weights=clus["pw"].values, at="overall"
)

se_naive = float(m_naive.dydx("meals").std_error)
se_cluster = float(m_cluster.dydx("meals").std_error)
print(f"naïve SE (ignores clustering): {se_naive:.3f}")
print(f"cluster-robust survey SE:      {se_cluster:.3f}")
print(f"design effect (variance ratio): "
      f"{(se_cluster / se_naive) ** 2:.1f}x")

naïve SE (ignores clustering): 0.035
cluster-robust survey SE:      0.302
design effect (variance ratio): 73.5x


In [8]:
fit_u = smf.glm("api00 ~ meals + ell", data=strat).fit()

m_posthoc = Margins(
    fit_u, survey_design=design, weights=strat["pw"].values, at="overall"
)
print(m_posthoc.dydx("meals").summary())

            Margins Result (delta, level=0.95)            
       estimate  std err        z  P>|z|  [95% Conf. Int.]
----------------------------------------------------------
meals   -2.8639   0.3145  -9.1056  0.000  -3.4803, -2.2475

n = 200
κ: 0.000
Delta-vs-sim disagreement: 1.947%


In [9]:
m_boot = Margins(
    fit_c,
    survey_design=design_c,
    weights=clus["pw"].values,
    at="overall",
    method="bootstrap",
    n_boot=400,
    rng_seed=0,
)
print(f"linearization SE: {se_cluster:.3f}")
print(f"bootstrap SE:     {float(m_boot.dydx('meals').std_error):.3f}")

linearization SE: 0.302


bootstrap SE:     0.383


In [10]:
def row(label, result):
    r = result.dydx("meals")
    return {"approach": label,
            "estimate": float(r.estimate),
            "std_error": float(r.std_error)}

summary = pd.DataFrame([
    row("apistrat: weighted (SRS SE)", m_weighted),
    row("apistrat: survey linearization", m_survey),
    row("apistrat: post-hoc unweighted", m_posthoc),
    row("apiclus1: naïve (ignores clusters)", m_naive),
    row("apiclus1: survey linearization", m_cluster),
    row("apiclus1: survey bootstrap", m_boot),
])
print(summary.round(3).to_string(index=False))
print(f"\npopulation truth slope on meals: {TRUTH_AME:.3f}")

                          approach  estimate  std_error
       apistrat: weighted (SRS SE)    -3.111      0.047
    apistrat: survey linearization    -3.111      0.276
     apistrat: post-hoc unweighted    -2.864      0.315
apiclus1: naïve (ignores clusters)    -3.146      0.035
    apiclus1: survey linearization    -3.146      0.302
        apiclus1: survey bootstrap    -3.146      0.383

population truth slope on meals: -2.963
